In [1]:
%matplotlib ipympl

import h5py
import matplotlib.pyplot as plt
import multiprocessing as mp
import numba
import numpy as np
import obspy
import os
import scipy.signal
import scipy.spatial
import tqdm

DTYPE_INT  = np.int32
DTYPE_REAL = np.float64

In [329]:
# Data are assumed row vectors!
PROGRESS_BAR_MESSAGE_LENGTH = 20
PROGRESS_BAR_TOTAL_LENGTH = 80

class FastMap(object):


    def __init__(self, path, kdim, distance, sampling_rate=100, mode="w", overwrite=False):
        self._kdim = kdim
        self._sampling_rate = sampling_rate
        self._init_hdf5(path, mode, overwrite=overwrite)
        self._kdtree = None
        group = self.hdf5.require_group("waveforms")
        self._waveforms = WaveformGroup(group.id)
        self._distance = distance

    
    @property
    def image(self):
        if "image" in self.hdf5:
            return (self.hdf5["image"])
        else:
            return (
                self.hdf5.create_dataset(
                    "/image",
                    shape=(self.library_size+1, self.kdim),
                    maxshape=(None, self.kdim),
                    dtype=DTYPE_REAL,
                    fillvalue=np.nan
                )
            )

    @property
    def kdim(self):
        return (self._kdim)
    
    @property
    def kdtree(self):
        if self._kdtree == None:
            self._kdtree = scipy.spatial.cKDTree(self.image[:-1])
        return (self._kdtree)
    
    @property
    def hdf5(self):
        return (self._hdf5)
    
    @property
    def library_size(self):
        """
        [Read only] The number of time series in the library.
        """
        return (len(self.waveforms))
    
    @property
    def pivot(self):
        return (
            self.hdf5.require_dataset(
                "/pivot", 
                shape=(2, self.kdim),
                dtype=DTYPE_INT
            )
        )
    
    @property
    def sampling_rate(self):
        return (self._sampling_rate)
    
    @property
    def waveforms(self):
        return (self._waveforms)


    def __del__(self):
        self.hdf5.close()


    def __enter__(self):
        return (self)


    def __exit__(self, exc_type, exc_value, exc_traceback):
        pass
    
    def _append(self, data, labels, sampling_rate):
        """
        Append data to the library inventory. This does not perform the
        actual embedding into k-dimensional Euclidean space.
        
        Arguments
        data - A single np.ndarray.
        """
        
        id = str(self.library_size)
        dataset = self.waveforms.create_dataset(id, data=data)
        dataset.attrs["labels"] = labels
        
        return (True)
    
    
    def _embed(self, ik):
        """
        Recursive function to embed the library data into k-dimensional
        Euclidean space.
        """

        if ik >= self.kdim:
            return (True)
        
        # Choose the pivot objects.
        idx_b = np.random.choice(np.arange(len(self.waveforms)-1))
        b = self.waveforms[idx_b]
        image_b = self.image[idx_b]
    
        idx_last_a = None
        idx_last_b = idx_b
        while True:
            idx_a = self.furthest(b, image_b, k=ik)
            a = self.waveforms[idx_a]
            image_a = self.image[idx_a]
            
            idx_b = self.furthest(a, image_a, k=ik)
            b = self.waveforms[idx_b]
            image_b = self.image[idx_b]
            
            if idx_a == idx_last_a and idx_b == idx_last_b:
                break
            idx_last_a = idx_a
            idx_last_b = idx_b
        
        # Record the names of the pivot objects.
        self.pivot[0, ik] = idx_a
        self.pivot[1, ik] = idx_b
        
        # Project all objects onto line between objects a and b.
        d_ab = reduced_distance((self._distance, a, b, image_a, image_b, ik))
        
        if d_ab == 0:
            self.image[:, icol] = 0
            return (True)
        
        desc = f"Updating image along dimension {ik}"
        pbar = tqdm.tqdm(
            total=len(self.waveforms),
            desc=f"{desc:{PROGRESS_BAR_MESSAGE_LENGTH}s}",
            ncols=PROGRESS_BAR_TOTAL_LENGTH
        )
        def update_image(name, i, ik=ik, pbar=pbar):
            irow = int(name)
            d_ai = self.distance(a, i, k=ik)
            d_bi = self.distance(b, i, k=ik)
            xi = (d_ai**2 + d_ab**2 - d_bi**2) / (2 * d_ab)
            self.image[irow, ik] = xi
            pbar.update(n=1)
        
        self.waveforms.visititems(update_image)
        pbar.close()
        
        return (self._embed(ik+1))
        
    
    def _init_hdf5(self, path, mode, overwrite=False):
        """
        Initialize the HDF5 backend.
        """
        
        if os.path.exists(path) and mode == "w" and not overwrite:
            raise (IOError(f"{path} already exists."))
        self._hdf5 = h5py.File(path, mode=mode)
        

    def append(self, data, labels, sampling_rate):
        """
        Append data to the library inventory. This does not perform the
        actual embedding into k-dimensional Euclidean space.
        
        Arguments
        data - A single np.ndarray or a list of np.ndarrays.
        """
        
        # Extend this to allow for multiple pieces of data to be added
        # with one call.
        self._append(data, labels, sampling_rate)
        
        return (True)
    

    def closest(self, b, k):
        """
        Return the name of the object closest to b.
        """
        
        self._closest_name, self._closest_dist = None, np.inf
        
        def _closest(name, a, b=b, k=k):
            d = self.distance(a, b, k=k)
            if d < self._closest_dist:
                self._closest_name = name
                self._closest_dist = d

        self.waveforms.visititems(_closest)
        
        closest_name = self._closest_name
        del (self._closest_name, self._closest_dist)
        return (closest_name)
    
    
    def distance(self, a, b, image_a, image_b, k=0):
        """
        deprecated
        """
        
        dist = self._distance(a, b)
        
        for ik in range(k):
            xa = image_a[ik]
            xb = image_b[ik]
            dist = dist**2 - (xa - xb)**2
            
            # This is a hack for distance functions that don't satisfy
            # the triangle inequality.
            dist = max(dist, 0)
            
            dist = np.sqrt(dist)
            
        return (dist)


    def embed(self):
        """
        Embed the library data into k-dimensional Euclidean space.
        """
        
        return_value = self._embed(0)
        
        return (return_value)

    
    def furthest(self, b, image_b, k):
        """
        Return the name of the object furthest from b.
        """

        with mp.Pool() as pool:
            dist = pool.map(
                reduced_distance,
                ((self._distance, self.waveforms[i][:], b[:], self.image[i][:], image_b[:], k) for i in range(len(self.waveforms)-1))
            )
        return (np.argmax(dist))
    
   
    def query(self, q, *args, **kwargs):

        q = self.waveforms.create_dataset("-1", data=q)
        
        for ik in range(self.kdim):

            # Retrieve the names of the pivot objects.
            a_name = self.pivot[0, ik] 
            b_name = self.pivot[1, ik]
            
            # Retrieve the pivot objects
            a = self.waveforms[a_name]
            b = self.waveforms[b_name]

            d_ab = self.distance(a, b, k=ik)
            
            if d_ab == 0:
                image[ik] = 0
                continue

            # Project query object onto line between objects a and b.
            d_aq = self.distance(a, q, k=ik)
            d_bq = self.distance(b, q, k=ik)
            xq = (d_aq**2 + d_ab**2 - d_bq**2) / (2 * d_ab)
            self.image[-1, ik] = xq
            
        del (self.waveforms["-1"])

        return (self.kdtree.query(self.image[-1], *args, **kwargs))


class WaveformGroup(h5py.Group):
    """
    Accessor class to abstract waveform access pattern.

    This provides functionality to access waveforms using integer
    indices.
    """
    
    def __getitem__(self, key):
        return(super().__getitem__(str(key)))
            

def correlate(a, b):
    """
    Return the normalized cross-correlation of a and b.
    
    a and b must be either 1D arrays or 2D arrays with shapes 
    (NDIM, NPTS_A) and (NDIM, NPTS_B).
    """

    a = np.atleast_2d(a)
    b = np.atleast_2d(b)
    
    assert a.shape[0] == b.shape[0]
    
    # Let a be the shorter of the two traces.
    if b.shape[1] < a.shape[1]:
        tmp = a
        a = b
        b = tmp
    
    m = a.shape[1]
    n = b.shape[1]
    
    df_a = pd.DataFrame(a).T
    df_b = pd.DataFrame(b).T
    
    df_a = df_a - df_a.mean()
    df_b = df_b - df_b.mean()
    
    if m == n:
        norm = m * df_a.std() * df_b.std()
    else:
        repeat = np.concatenate(
            [
                np.repeat(0, m-1),
                [m],
                np.repeat(1, n-m-1),
                [m]
            ]
        )
        var_b = df_b.rolling(m).var()
        var_b = var_b.loc[var_b.index.repeat(repeat)]
        norm = np.sqrt(n * var_b) * np.sqrt(n * df_a.var())
        norm = norm.reset_index(drop=True)

    corr = np.empty((a.shape[0], a.shape[1]+b.shape[1]-1))

    for i in range(a.shape[0]):
        corr[i] = np.correlate(a[i], b[i], "full") / norm[i]
            
    return (corr)


def distance(a, b):
    """
    Return the distance between a and b.
    """
    
    corr = correlate(a, b)
    ndim = corr.shape[0]

    corr = np.sum(np.abs(corr), axis=0)
    cmax = np.max(corr)
    
    dist = (ndim - cmax) / ndim
    
    return (dist)


def reduced_distance(args):
    distance, a, b, image_a, image_b, k = args
    dist = distance(a, b)
    
    for ik in range(k):
        xa = image_a[ik]
        xb = image_b[ik]
        dist = dist**2 - (xa - xb)**2

        # This is a hack for distance functions that don't satisfy
        # the triangle inequality.
        dist = max(dist, 0)

        dist = np.sqrt(dist)
        
    return (dist)

In [330]:
training_frac = 0.8
network, station = "PB", "B049"

dataframe = pd.read_csv("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/STEAD/chunk2.csv")
dataframe = dataframe.set_index(["network_code", "receiver_code"])
dataframe = dataframe.sort_index()
dataframe = dataframe.loc[(network, station)]
dataframe = dataframe.reset_index()

dataframe = dataframe.sample(n=32)
print(len(dataframe))

training_data = dataframe.sample(frac=training_frac)
test_data = dataframe.drop(training_data.index)

try:
    fastmap.hdf5.close()
except:
    pass

fastmap = FastMap(f"{network}.{station}.h5", 8, distance, overwrite=True)

def get_waveforms(label, event, f5):
    trace_name = event["trace_name"]
    handle = f"data/{trace_name}"
    dataset = f5[handle]
    attrs = dataset.attrs
    
#     phase = "P" if label == "N" else label
#     ioff = 300 if label == "N" else 0
#     idx = attrs[f"{phase.lower()}_arrival_sample"]
#     idx_start = int(idx - ioff - 100)
#     idx_end = int(idx - ioff + 200)

    idx_p = attrs["p_arrival_sample"]
    idx_s = attrs["s_arrival_sample"]
    smp = idx_s - idx_p
    idx_start = int(idx_p - min(smp/4, 100)) if label == "P" else int(idx_s - min(smp/4, 100)) if label == "S" else int(idx_p - 2*smp)
    idx_end = int(idx_p + min(smp/2, 300)) if label == "P" else int(idx_s + min(smp/2, 300)) if label == "S" else int(idx_p - smp)

    data = np.empty_like(dataset)
    
    for i in range(3):
        trace = obspy.Trace(data=dataset[:, i])
        trace.stats.sampling_rate = 100
        trace.filter("bandpass", freqmin=2, freqmax=20)
        data[:, i] = trace.data
        
    trace = data[idx_start: idx_end].T
    
    if trace.size == 0:
        return (None)
    
    trace /= np.atleast_2d(np.max(np.abs(trace), axis=1)).T
    
    return (trace)

with h5py.File("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/STEAD/chunk2.hdf5", mode="r") as f5:
    for index, event in training_data.iterrows():
        for label in ("P", "S", "N"):
            trace = get_waveforms(label, event, f5)
            if trace is None:
                continue
            fastmap._append(trace, label, 100)
            

%time fastmap.embed()

32


Updating image along dimension 0:   0%|                  | 0/67 [00:00<?, ?it/s]

TypeError: distance() missing 2 required positional arguments: 'image_a' and 'image_b'

In [311]:
a = fastmap.waveforms[0]
image_a = fastmap.image[0]

In [312]:
i = fastmap.furthest(a, image_a, 0)
b = fastmap.waveforms[i]
image_b = fastmap.image[i]

In [317]:
distance(a, b), reduced_distance((distance, a, b, image_a, image_b, 0)), reduced_distance((distance, b, a, image_b, image_a, 0))

(0.92236052334262764, 0.92236052334262764, 0.92236052334262764)

In [ ]:
distance(a, b), reduced_distance

In [308]:
norm

array([[ -1.24940482e-04,  -1.78910204e-04,  -1.10309891e-04, ...,
         -1.05316642e-04,  -2.80520468e-05,  -2.38454026e-05],
       [  1.47524919e-04,   2.82542303e-04,   4.56109178e-04, ...,
          2.51257674e-05,  -2.92223097e-05,   3.61499081e-06],
       [ -6.89597387e-05,  -1.65458173e-04,  -2.08281857e-04, ...,
         -6.48154632e-05,  -1.77391409e-05,   1.60273263e-05]])

In [ ]:
import mpl_toolkits.mplot3d.axes3d

plt.close("all")
fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(1, 1, 1, projection="3d")
for label in ("P", "S", "N"):
    idxs = [int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"][0] == label]
    idxs = sorted(idxs)
    ax.scatter(
        fastmap.image[idxs, 0],
        fastmap.image[idxs, 1],
        fastmap.image[idxs, 2],
        s=8,
        linewidth=0,
        label=label
    )

In [ ]:
ax.set_xlim(0.4, 0.6)
ax.set_ylim(0.4, 0.6)
ax.set_zlim(0.4, 0.6)
# ax.set_xticks(np.arange(*ax.get_xlim(), 0.05))
# ax.set_yticks(np.arange(*ax.get_ylim(), 0.05))
# ax.set_zticks(np.arange(*ax.get_zlim(), 0.05))
# ax.set_xticklabels([])
# ax.set_yticklabels([])
# ax.set_zticklabels([])
# plt.savefig("/home/malcolmw/google_drive/malcolm.white@usc.edu/courses/PHYS-760/project/presentation/figures/3d_clusters.png", dpi=1080, bbox_inches="tight")

In [ ]:
sample = test_data.sample(n=1).iloc[0]

with h5py.File("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/STEAD/chunk2.hdf5", mode="r") as f5:
    q = None
    while q is None:
        label = np.random.choice(["P", "S", "N"])
        q = get_waveforms(label, sample, f5)

d, i = fastmap.query(q, k=50)
dist = [distance(q, fastmap.waveforms[idx]) for idx in i]
i = i[np.argmin(dist)]

# %time d, i = fastmap.query(q)
m = fastmap.waveforms[i]

print(np.min(dist), np.max(dist), np.max(np.abs(np.sum(correlate(q, m), axis=0)))/3)


plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(2, 1, 1)
for i in range(3):
    ax.plot(q[i]/np.max(np.abs(q[i])) + i)
ax.set_xlim(0, q.shape[1])
    
ax.text(0.05, 0.95, label, transform=ax.transAxes, va="top", ha="left")
ax = fig.add_subplot(2, 1, 2)
ax.text(0.05, 0.95, m.attrs['labels'], transform=ax.transAxes, va="top", ha="left")
for i in range(3):
    ax.plot(m[i]/np.max(np.abs(m[i])) + i)
ax.set_xlim(0, m.shape[1])

In [ ]:
with h5py.File("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/STEAD/chunk2.hdf5", mode="r") as f5:
    p = get_waveforms("P", sample, f5)
    s = get_waveforms("S", sample, f5)

In [ ]:
plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(2, 1, 1)
ax.plot(p.T)
ax = fig.add_subplot(2, 1, 2)
ax.plot(s.T)

In [ ]:
plt.close("all")
fig = plt.figure(figsize=(12, 12))

import matplotlib as mpl

gs = mpl.gridspec.GridSpec(nrows=fastmap.kdim, ncols=fastmap.kdim)

p_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "P"])
s_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "S"])
n_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "N"])

for ix in range(fastmap.kdim):
    for iy in range(ix+1, fastmap.kdim):
        ax = fig.add_subplot(gs[ix, iy])
        for idxs in (p_idxs, s_idxs, n_idxs):
            ax.scatter(
                fastmap.image[idxs, ix],
                fastmap.image[idxs, iy],
                s=1,
                linewidth=0
            )
        break
    break
    
#         ax.set_xlim(0, 1)
#         ax.set_ylim(0, 1)

In [ ]:
import KDEpy as kp

In [ ]:
kde = kp.FFTKDE().fit(fastmap.image[idxs][[ix, iy]])

In [ ]:
xy, amp = kde.evaluate()

In [ ]:
data = fastmap.image[idxs][:, [ix, iy]]
kde = kp.FFTKDE().fit(data)
xy, amp = kde.evaluate()
xy = xy.reshape(32, 32, 2)
amp = amp.reshape(32, 32)

In [ ]:
import seaborn as sns

In [ ]:
ix, iy = 0, 2
p_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "P"])
s_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "S"])
# n_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "N"])

plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
for idxs in (p_idxs, s_idxs):#, n_idxs):
    data = fastmap.image[idxs][:, [ix, iy]]
    sns.kdeplot(data[:, 0], data[:, 1], levels=3)
    
xmin, xmax = np.quantile(fastmap.image[:, ix], [0.02, 0.98])
ymin, ymax = np.quantile(fastmap.image[:, iy], [0.02, 0.98])
dx = xmax - xmin
dy = ymax - ymin

ax.set_xlim(xmin - dx/10, xmax + dx/10)
ax.set_ylim(ymin - dy/10, ymax + dy/10)

In [ ]:
xmin, xmax = np.quantile(fastmap.image[:, 0], [0.02, 0.98])
ymin, ymax = np.quantile(fastmap.image[:, 1], [0.02, 0.98])
dx = xmax - xmin
dy = ymax - ymin

ax.set_xlim(xmin - dx/10, xmax + dx/10)
ax.set_ylim(ymin - dy/10, ymax + dy/10)

In [ ]:
ix, iy = 0, 1
p_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "P"])
s_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "S"])
n_idxs = sorted([int(key) for key in fastmap.waveforms.keys() if fastmap.waveforms[key].attrs["labels"] == "N"])

plt.close("all")
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
for idxs, color in zip((p_idxs, n_idxs), ("tab:blue", "tab:green")):
    data = fastmap.image[idxs][:, [ix, iy]]
    

In [ ]:
count = dict(PP=0, PS=0, PN=0, SP=0, SS=0, SN=0, NP=0, NS=0, NN=0)

with h5py.File("/home/malcolmw/google_drive/malcolm.white@usc.edu/data/wfs/STEAD/chunk2.hdf5", mode="r") as f5:
    for _, event in tqdm.tqdm(test_data.iterrows(), total=len(test_data)):
        for label0 in ("P", "S", "N"):
            q = get_waveforms(label0, event, f5)
            if q is None:
                continue
            d, i = fastmap.query(q, k=19)
            i0 = np.argmin([distance(q, fastmap.waveforms[idx]) for idx in i])
            m = fastmap.waveforms[i[i0]]
            label = m.attrs["labels"]
            if np.max(np.abs(np.sum(correlate(q, m), axis=0)))/3 < 0.25:
                label = "N"
            count[f"{label0}{label}"] += 1

In [ ]:
print(f"Precision P: = {count['PP'] / (count['PP'] + count['SP'] + count['NP'])}")
print(f"Precision S: = {count['SS'] / (count['SS'] + count['PS'] + count['NS'])}")
print(f"Precision N: = {count['NN'] / (count['NN'] + count['PN'] + count['SN'])}")
print(f"Recall P: = {count['PP'] / (count['PP'] + count['PS'] + count['PN'])}")
print(f"Recall S: = {count['SS'] / (count['SS'] + count['SP'] + count['SN'])}")
print(f"Recall N: = {count['NN'] / (count['NN'] + count['NP'] + count['NS'])}")
print()

In [ ]:
print(f"Precision P: = {count['PP'] / (count['PP'] + count['SP'] + count['NP'])}")
print(f"Precision S: = {count['SS'] / (count['SS'] + count['PS'] + count['NS'])}")
print(f"Precision N: = {count['NN'] / (count['NN'] + count['PN'] + count['SN'])}")
print(f"Recall P: = {count['PP'] / (count['PP'] + count['PS'] + count['PN'])}")
print(f"Recall S: = {count['SS'] / (count['SS'] + count['SP'] + count['SN'])}")
print(f"Recall S: = {count['NN'] / (count['NN'] + count['NP'] + count['NS'])}")
print()

In [ ]:
count